In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime
from google.cloud import bigquery
from google.cloud import storage

# Set project variables
PROJECT_ID = "telco-portfolio"
BUCKET_NAME = f"telco-portfolio-bucket-{datetime.now().strftime('%Y%m%d')}"
DATASET_ID = "telco_dataset"
TABLE_ID = "raw_churn"

print(f"✅ Project: {PROJECT_ID}")
print(f"✅ Bucket: {BUCKET_NAME}")
print(f"✅ Dataset: {DATASET_ID}")

✅ Project: telco-portfolio
✅ Bucket: telco-portfolio-bucket-20260601
✅ Dataset: telco_dataset


In [2]:
# Cell 2: GCP Authentication - Complete & Clean
import subprocess
import sys
import os

print("🔐 Setting up GCP authentication...")

# 1. Set project aktif
result = subprocess.run(["gcloud", "config", "set", "project", "telco-portfolio"], 
                        capture_output=True, text=True)
if result.returncode == 0:
    print("✅ Project set to: telco-portfolio")

# 2. Set quota project untuk ADC (Application Default Credentials)
result = subprocess.run(["gcloud", "auth", "application-default", "set-quota-project", "telco-portfolio"],
                        capture_output=True, text=True)
if result.returncode == 0:
    print("✅ Quota project set to: telco-portfolio")
else:
    print("⚠️ Perlu login ulang, jalankan: gcloud auth application-default login")

# 3. Cek akun yang aktif
result = subprocess.run(["gcloud", "auth", "list", "--format=value(account,status)"],
                        capture_output=True, text=True)
print("\n📋 Credentialed Accounts:")
print(result.stdout)

# 4. Verifikasi project aktif
result = subprocess.run(["gcloud", "config", "get-value", "project"],
                        capture_output=True, text=True)
current_project = result.stdout.strip()
print(f"📁 Active project: {current_project}")

# 5. Verifikasi quota project di ADC (tanpa warning)
from google.auth import default
from google.auth.transport.requests import Request

try:
    credentials, project = default()
    quota_project = credentials.quota_project_id if hasattr(credentials, 'quota_project_id') else "Not set"
    print(f"🎫 ADC Quota Project: {quota_project}")
    
    if quota_project == "telco-portfolio" or project == "telco-portfolio":
        print("✅ GCP configuration complete - No warnings!")
    else:
        print("⚠️ Masih ada mismatch, jalankan ulang step 2")
except Exception as e:
    print(f"⚠️ ADC check: {e}")

print("\n🎯 Semua konfigurasi GCP selesai dengan bersih!")

🔐 Setting up GCP authentication...
✅ Project set to: telco-portfolio
✅ Quota project set to: telco-portfolio

📋 Credentialed Accounts:
studyburhanudin@gmail.com	*

📁 Active project: telco-portfolio
🎫 ADC Quota Project: telco-portfolio
✅ GCP configuration complete - No warnings!

🎯 Semua konfigurasi GCP selesai dengan bersih!


In [3]:
# Buat bucket (jika belum ada)
!gsutil ls gs://{BUCKET_NAME} 2>/dev/null || gsutil mb -l us-central1 gs://{BUCKET_NAME}

print(f"✅ Bucket ready: gs://{BUCKET_NAME}")

gs://telco-portfolio-bucket-20260601/data/
✅ Bucket ready: gs://telco-portfolio-bucket-20260601


In [4]:
# Cell 4: Download dataset dengan URL yang valid
import urllib.request
import os

# Gunakan URL dari IBM Cloud Pak for Data (masih aktif)
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
output_file = "telco_churn.csv"

print("📥 Downloading Telco Customer Churn dataset...")
print(f"📡 URL: {url}")

try:
    # Download file
    urllib.request.urlretrieve(url, output_file)
    
    # Cek ukuran file
    file_size = os.path.getsize(output_file)
    print(f"✅ Download successful!")
    print(f"📦 File size: {file_size:,} bytes ({file_size/1024:.1f} KB)")
    
    # Baca dan tampilkan preview
    import pandas as pd
    df = pd.read_csv(output_file)
    print(f"📊 Total rows: {len(df):,}")
    print(f"📊 Total columns: {len(df.columns)}")
    print("\n🔍 Preview first 2 rows:")
    print(df.head(2))
    
except Exception as e:
    print(f"❌ Download failed: {e}")
    print("\n🔄 Mencoba URL alternatif...")
    
    # Fallback ke URL Watson Analytics
    url2 = "https://community.watsonanalytics.com/wp-content/uploads/2015/03/WA_Fn-UseC_-Telco-Customer-Churn.csv"
    try:
        urllib.request.urlretrieve(url2, output_file)
        file_size = os.path.getsize(output_file)
        print(f"✅ Download successful with alternative URL!")
        print(f"📦 File size: {file_size:,} bytes ({file_size/1024:.1f} KB)")
    except Exception as e2:
        print(f"❌ Alternative URL also failed: {e2}")
        print("\n💡 Manual download required:")
        print("   Download from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn")
        print("   Then upload to this notebook folder.")

📥 Downloading Telco Customer Churn dataset...
📡 URL: https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv
✅ Download successful!
📦 File size: 970,457 bytes (947.7 KB)
📊 Total rows: 7,043
📊 Total columns: 21

🔍 Preview first 2 rows:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   

  TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling  \
0          No          No              No  Month-to-month              Yes   
1          No          No              No        One year               No   

      Payme

In [5]:
# Cell 5: Upload ke Cloud Storage
from google.cloud import storage
from datetime import datetime

# Inisialisasi bucket name (sama seperti sebelumnya)
BUCKET_NAME = f"telco-portfolio-bucket-{datetime.now().strftime('%Y%m%d')}"
print(f"📦 Target bucket: gs://{BUCKET_NAME}")

# Upload file
client = storage.Client(project="telco-portfolio")
bucket = client.bucket(BUCKET_NAME)
blob = bucket.blob("data/raw/telco_churn.csv")

# Upload file yang sudah didownload
blob.upload_from_filename("telco_churn.csv")

print(f"✅ Upload successful!")
print(f"📍 Location: gs://{BUCKET_NAME}/data/raw/telco_churn.csv")

# Verifikasi
!gsutil ls -lh gs://{BUCKET_NAME}/data/raw/telco_churn.csv

📦 Target bucket: gs://telco-portfolio-bucket-20260601
✅ Upload successful!
📍 Location: gs://telco-portfolio-bucket-20260601/data/raw/telco_churn.csv
947.71 KiB  2026-06-01T14:17:13Z  gs://telco-portfolio-bucket-20260601/data/raw/telco_churn.csv
TOTAL: 1 objects, 970457 bytes (947.71 KiB)


In [6]:
# Inisialisasi BigQuery client
client = bigquery.Client(project=PROJECT_ID)

# Buat dataset
dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
dataset = bigquery.Dataset(dataset_ref)
dataset.location = "us-central1"

try:
    client.create_dataset(dataset)
    print(f"✅ Dataset {DATASET_ID} created")
except Exception as e:
    print(f"ℹ️ Dataset already exists: {e}")

ℹ️ Dataset already exists: 409 POST https://bigquery.googleapis.com/bigquery/v2/projects/telco-portfolio/datasets?prettyPrint=false: Already Exists: Dataset telco-portfolio:telco_dataset


In [7]:
# Load data dari GCS ke BigQuery
uri = f"gs://{BUCKET_NAME}/data/raw/telco_churn.csv"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
)

load_job = client.load_table_from_uri(
    uri,
    f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}",
    job_config=job_config,
)

load_job.result()  # Tunggu selesai

# Verifikasi
table = client.get_table(f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}")
print(f"✅ Loaded {table.num_rows} rows to {DATASET_ID}.{TABLE_ID}")

✅ Loaded 14086 rows to telco_dataset.raw_churn


In [8]:
# Lihat sample data
query = f"""
SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
LIMIT 5
"""

df_sample = client.query(query).to_dataframe()
df_sample.head()

/Users/macos/Study_burhanudin_2025/GCP/telco-ai-portfolio/venv_telco/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,9426-SXNHE,Female,0,False,False,2,True,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,False,Bank transfer (automatic),18.75,53.15,False
1,3806-YAZOV,Female,0,False,False,3,True,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,False,Mailed check,18.80,56,False
2,3387-PLKUI,Female,0,True,True,13,True,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,False,Mailed check,18.80,251.25,False
3,8992-CEUEN,Female,0,False,False,1,True,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,False,Electronic check,18.85,18.85,False
4,0620-XEFWH,Male,0,True,True,4,True,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,False,Mailed check,18.85,84.2,False
